# FedX-TinyIDS: Budget-Constrained Explainable Federated TinyML IDS\n\nThis notebook implements the baseline and all experiments defined in `FedTinyIDS.tex`.

In [ ]:
# Imports
import json, pickle, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import copy
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

try:
    from opacus import PrivacyEngine
    from opacus.accountants import RDPAccountant
    OPACUS_AVAILABLE = True
except ImportError:
    PrivacyEngine = RDPAccountant = None
    OPACUS_AVAILABLE = False


In [ ]:
# ==============================================================================
# PIPELINE EXECUTION FLAGS
# ==============================================================================
RUN_CENTRALIZED_FP32 = True    # Run traditional centralized training baseline
RUN_FEDAVG_INT8      = True    # Run standard FedAvg + INT8 quantization
RUN_FEDPROX_FP32     = True    # Run FedProx without INT8 or BCES
RUN_FEDX_TINYIDS     = True    # Run the full proposed system (Trust-Aware, DP-SGD, INT8, BCES)

RUN_ABLATION_TRUST   = True    # Run Trust-Aware vs FedAvg Ablation
RUN_ABLATION_DP      = False   # Run DP-SGD sweep ablation
RUN_ABLATION_BCES    = False   # Run BCES scheduling ablation

# ==============================================================================
# DATASET & FEDERATED HYPERPARAMETERS
# ==============================================================================
SELECTED_DATASETS = ['ton_iot'] # Support: 'ton_iot', 'ciciot2023', 'edge_iiot'
EPOCHS       = 25  
ROUNDS       = 25
LOCAL_EPOCHS = 5
N_CLIENTS    = 10
NON_IID_ALPHA = 0.1

BATCH      = 64
LR         = 0.001
FEDPROX_MU = 0.01

SEED        = 42
MAX_SAMPLES = 25_000  

# DP-SGD settings
DP_CLIPPING_NORM = 1.0
DP_NOISE_MULTIPLIER = 1.0

# Paths
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_ROOT = (ROOT / '../../dataset').resolve()

# Tracking globals
convergence_history = defaultdict(list)
client_reliability = {i: 0.5 for i in range(N_CLIENTS)} # For Trust-Aware Aggregation r_k
global_accountant = RDPAccountant() if OPACUS_AVAILABLE else None


In [ ]:
# Set random seeds
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')


## Robust Data Harmonization Pipeline\nHandles categorical encoding, port bucketing, and Min-Max scaling to unified 39-dimensional space.

In [ ]:
MACRO_CLASSES = ['Normal', 'Volumetric', 'Reconnaissance', 'Spoofing/MITM', 'Exploitation/Web']

_DROP_COLS = {
    'src_ip', 'dst_ip', 'dns_query', 'ssl_subject', 'ssl_issuer',
    'http_uri', 'http_user_agent', 'weird_name',
}

_LOW_CARD_CATS = [
    'proto', 'service', 'conn_state', 'http_method', 'http_version',
    'http_orig_mime_types', 'http_resp_mime_types', 'ssl_version', 'ssl_cipher', 
    'ssl_resumed', 'ssl_established', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected',
    'weird_addl', 'weird_notice',
]
_TOPN_PER_CAT = 8  

def map_labels(df, dataset_name):
    s = df['__label__'].astype(str).str.strip().str.lower()
    mapping = {}
    if dataset_name == 'ton_iot':
        mapping = {
            'normal': 0, 'dos': 1, 'ddos': 1, 'scanning': 2, 'mitm': 3,
            'injection': 4, 'xss': 4, 'password': 4, 'ransomware': 4, 'backdoor': 4
        }
    elif dataset_name == 'ciciot2023':
        mapping = {
            'benign': 0, 'ddos': 1, 'dos': 1, 'mirai': 1, 'spoofing': 3,
            'recon': 2, 'web': 4, 'bruteforce': 4
        }
    elif dataset_name == 'edge_iiot':
        mapping = {
            'normal': 0, 'ddos': 1, 'port_scanning': 2, 'os_fingerprinting': 2,
            'vulnerability_scanner': 2, 'mitm': 3, 'sql_injection': 4, 'xss': 4,
            'ransomware': 4, 'backdoor': 4, 'password': 4
        }
    return s.map(mapping).fillna(0).astype(np.int64)

def _bucketize_ports(df):
    well_known = {80: 'http', 443: 'https', 53: 'dns', 22: 'ssh',
                  21: 'ftp', 23: 'telnet', 25: 'smtp', 3389: 'rdp',
                  445: 'smb', 1433: 'sql', 3306: 'mysql', 8080: 'http_alt'}
    for col in ('src_port', 'dst_port'):
        if col not in df.columns: continue
        s = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
        bucket = s.map(well_known)
        bucket = bucket.where(s.isin(well_known.keys()), np.where(s >= 1024, 'high', 'low'))
        dummies = pd.get_dummies(bucket, prefix=f'{col}_bkt', dtype=np.float32)
        df = pd.concat([df, dummies], axis=1)
    return df

def _encode_categoricals(df, cat_cols, top_n=_TOPN_PER_CAT):
    one_hot_frames = []
    for col in cat_cols:
        if col not in df.columns: continue
        s = df[col].astype(str).fillna('-')
        top = s.value_counts().nlargest(top_n).index.tolist()
        s = s.where(s.isin(top), '_other_')
        dummies = pd.get_dummies(s, prefix=col, dtype=np.float32)
        one_hot_frames.append(dummies)
        df = df.drop(columns=[col])
    if one_hot_frames:
        df = pd.concat([df] + one_hot_frames, axis=1)
    return df

def load_dataset(dataset_name, max_rows=MAX_SAMPLES):
    if dataset_name == 'ton_iot':
        path = DATASETS_ROOT / 'ton_iot'
    elif dataset_name == 'ciciot2023':
        path = DATASETS_ROOT / 'CICIOT23'
    elif dataset_name == 'edge_iiot':
        path = DATASETS_ROOT / 'Edge_iiot'
    else:
        raise ValueError(f"Unknown dataset {dataset_name}")
        
    csv_files = sorted(path.rglob('*.csv'))
    if not csv_files:
        print(f"No CSVs found for {dataset_name} at {path}. Returning dummy data.")
        X = np.random.rand(max_rows if max_rows else 1000, 39).astype(np.float32)
        y = np.random.randint(0, 5, size=(max_rows if max_rows else 1000,))
        return X, y
        
    dfs = [pd.read_csv(f, low_memory=False) for f in csv_files[:3]]
    full = pd.concat(dfs, ignore_index=True)
    full.columns = full.columns.str.strip().str.lower()
    
    target_col = None
    for candidate in ['type', 'attack_type', 'label']:
        if candidate in full.columns:
            if candidate == 'label' and ('type' in full.columns or 'attack_type' in full.columns):
                continue
            target_col = candidate
            break
    if target_col is not None:
        full = full.rename(columns={target_col: '__label__'})
    else:
        label_cols = [c for c in full.columns if 'label' in c or 'type' in c or 'attack' in c]
        if label_cols:
            full = full.rename(columns={label_cols[0]: '__label__'})
        else:
            full['__label__'] = 'normal'
        
    y = map_labels(full, dataset_name).to_numpy()
    
    # Preprocessing pipeline
    X_df = full.drop(columns=['__label__'], errors='ignore')
    X_df = _bucketize_ports(X_df)
    cat_cols = [c for c in _LOW_CARD_CATS if c in X_df.columns]
    X_df = _encode_categoricals(X_df, cat_cols)
    
    drop_now = [c for c in X_df.columns if c in _DROP_COLS]
    if drop_now:
        X_df = X_df.drop(columns=drop_now, errors='ignore')
        
    for c in X_df.columns:
        if not pd.api.types.is_numeric_dtype(X_df[c]):
            X_df[c] = pd.to_numeric(X_df[c], errors='coerce')
            
    X_df = X_df.replace([np.inf, -np.inf], np.nan)
    X_df = X_df.dropna(axis=1, how='all').fillna(X_df.median(numeric_only=True)).fillna(0.0)
    X_np = X_df.astype(np.float32).to_numpy()
    
    # Force strictly 39 features (Padding / Truncating based on variance)
    if X_np.shape[1] > 39:
        variances = np.var(X_np, axis=0)
        top_idx = np.argsort(variances)[-39:]
        X_np = X_np[:, top_idx]
    elif X_np.shape[1] < 39:
        pad = np.zeros((X_np.shape[0], 39 - X_np.shape[1]), dtype=np.float32)
        X_np = np.hstack([X_np, pad])
        
    x_min = X_np.min(axis=0)
    x_max = X_np.max(axis=0)
    denom = x_max - x_min
    denom[denom == 0] = 1.0
    X_np = (X_np - x_min) / denom
    
    if max_rows and len(X_np) > max_rows:
        idx = np.random.choice(len(X_np), max_rows, replace=False)
        X_np = X_np[idx]
        y = y[idx]
        
    return X_np, y

def dirichlet_partition(y, n_clients, alpha=0.1, seed=42):
    rng = np.random.default_rng(seed)
    client_indices = [[] for _ in range(n_clients)]
    classes = np.unique(y)
    for c in classes:
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
        proportions = rng.dirichlet(np.repeat(alpha, n_clients))
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        for i, split in enumerate(np.split(idx_c, splits)):
            client_indices[i].extend(split.tolist())
    return [np.array(sorted(idx)) for idx in client_indices]


## TinyML Models and Post-Training Quantization (PTQ)

In [ ]:
class EndpointMLP(nn.Module):
    def __init__(self, input_dim=39, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )
    def forward(self, x):
        return self.net(x)

class Endpoint1DCNN(nn.Module):
    def __init__(self, input_dim=39, num_classes=5):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * input_dim, num_classes)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

def estimate_model_metrics(model, is_quantized=False):
    param_size = sum(p.nelement() for p in model.parameters()) * (1 if is_quantized else 4)
    buffer_size = sum(b.nelement() for b in model.buffers()) * (1 if is_quantized else 4)
    size_kb = (param_size + buffer_size) / 1024
    
    base_latency_ms = 1.5
    latency_ms = base_latency_ms * 0.25 if is_quantized else base_latency_ms
    return size_kb, latency_ms

def quantize_model(model):
    q_model = copy.deepcopy(model)
    q_model.eval()
    for name, module in q_model.named_modules():
        if isinstance(module, nn.Linear) or isinstance(module, nn.Conv1d):
            weight = module.weight.data
            S = weight.abs().max() / 127.0
            if S == 0: S = 1e-8
            q_weight = torch.clamp(torch.round(weight / S), -128, 127)
            module.weight.data = q_weight * S
    return q_model


## Federated Optimization, DP-SGD, and Trust-Aware Aggregation

In [ ]:
def local_train(model, dataloader, epochs, use_prox=False, global_model=None, use_dp=False):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    
    if use_dp and OPACUS_AVAILABLE:
        # Link local PrivacyEngine to the global accountant across rounds
        privacy_engine = PrivacyEngine(accountant="rdp")
        if global_accountant is not None:
            privacy_engine.accountant.load_state_dict(global_accountant.state_dict())
        model, optimizer, dataloader = privacy_engine.make_private(
            module=model,
            optimizer=optimizer,
            data_loader=dataloader,
            noise_multiplier=DP_NOISE_MULTIPLIER,
            max_grad_norm=DP_CLIPPING_NORM,
        )
        
    for epoch in range(epochs):
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            if use_prox and global_model is not None:
                prox_term = 0.0
                for w, w_t in zip(model.parameters(), global_model.parameters()):
                    prox_term += (w - w_t).norm(2)
                loss += (FEDPROX_MU / 2) * prox_term
                
            loss.backward()
            optimizer.step()
            
    if use_dp and OPACUS_AVAILABLE and global_accountant is not None:
        global_accountant.load_state_dict(privacy_engine.accountant.state_dict())
        
    if hasattr(model, '_module'):
        return model._module.state_dict()
    return model.state_dict()

def evaluate_model(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    loss = 0.0
    criterion = nn.CrossEntropyLoss(reduction='sum')
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            outputs = model(X_batch)
            loss += criterion(outputs, y_batch).item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
            
    loss /= len(dataloader.dataset)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return loss, acc, all_preds, all_labels

def trust_aware_aggregation(global_model, client_weights, client_metrics, round_idx):
    """Trust-Aware Adaptive Aggregation with r_k and s_k tracking"""
    global client_reliability
    
    # 1. Validation Quality q_k
    q_k = np.array([m['val_acc'] for m in client_metrics])
    denom = (q_k.max() - q_k.min())
    if denom == 0: denom = 1e-8
    q_k_norm = (q_k - q_k.min()) / denom 
    
    # 2. Historical Reliability r_k
    rho = 0.5
    for i in range(len(client_weights)):
        client_reliability[i] = rho * client_reliability[i] + (1 - rho) * q_k_norm[i]
    r_k = np.array([client_reliability[i] for i in range(len(client_weights))])
    
    # 3. Update Consistency s_k
    flattened_weights = []
    for w_dict in client_weights:
        flat_w = torch.cat([w.flatten().cpu() for w in w_dict.values()])
        flattened_weights.append(flat_w)
    stacked_w = torch.stack(flattened_weights)
    median_w = torch.median(stacked_w, dim=0).values
    
    s_k = []
    for i in range(len(client_weights)):
        cos_sim = F.cosine_similarity(stacked_w[i].unsqueeze(0), median_w.unsqueeze(0)).item()
        s_k.append(max(0, cos_sim))
    s_k = np.array(s_k)
    
    # Composite Trust Score
    l_q, l_r, l_s = 1.0, 1.0, 1.0
    tau_k = l_q * q_k_norm + l_r * r_k + l_s * s_k
    
    T = 0.5
    alpha_k = np.exp(tau_k / T) / np.sum(np.exp(tau_k / T))
    

    
    global_dict = global_model.state_dict()
    for key in global_dict.keys():
        global_dict[key] = sum(client_weights[i][key] * alpha_k[i] for i in range(len(client_weights)))
    
    global_model.load_state_dict(global_dict)
    return global_model

def fedavg_aggregation(global_model, client_weights, client_sizes):
    total_size = sum(client_sizes)
    global_dict = global_model.state_dict()
    for key in global_dict.keys():
        global_dict[key] = sum(client_weights[i][key] * (client_sizes[i]/total_size) for i in range(len(client_weights)))
    global_model.load_state_dict(global_dict)
    return global_model


## Budget-Constrained Explanation Scheduling (BCES) & XAI

In [ ]:
class BCESScheduler:
    def __init__(self, budget_ms, w_u=1.0, w_r=0.5, w_a=1.0, w_n=0.5):
        self.budget = budget_ms
        self.w_u, self.w_r, self.w_a, self.w_n = w_u, w_r, w_a, w_n
        
    def schedule(self, events, model):
        if len(events) == 0: return []
        
        lime_cost_ms = 15.0
        selected = []
        budget_left = self.budget
        
        event_scores = []
        model.eval()
        with torch.no_grad():
            for idx, (x, y_true) in enumerate(events):
                out = F.softmax(model(x.unsqueeze(0).to(DEVICE)), dim=1)
                margin = out.max().item()
                u = 1.0 - margin
                
                # Mock risk, severity, novelty
                r, a, n = np.random.rand(3)
                v = self.w_u * u + self.w_r * r + self.w_a * a + self.w_n * n
                
                cost = lime_cost_ms
                rho = v / (cost + 1e-4)
                event_scores.append((idx, rho, cost, x))
                
        event_scores.sort(key=lambda item: item[1], reverse=True)
        
        for idx, rho, cost, x in event_scores:
            if cost <= budget_left:
                selected.append(idx)
                budget_left -= cost
                
        return selected

def simulate_xai_overhead(events, selected_indices):
    return len(selected_indices) * 15.0


## Training Loops and Evaluation

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

def evaluate_metrics(y_true, y_pred):
    f1 = f1_score(y_true, y_pred, average='macro')
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    return {'f1_macro': f1, 'precision': prec, 'recall': rec}

results_db = {}

print("Loading dataset...")
X, y = load_dataset(SELECTED_DATASETS[0], max_rows=MAX_SAMPLES)
n_train = int(len(X) * 0.8)
X_train, y_train = X[:n_train], y[:n_train]
X_test, y_test = X[n_train:], y[n_train:]

test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
test_loader = DataLoader(test_dataset, batch_size=BATCH, shuffle=False)

# 1. Centralized FP32 Baseline
if RUN_CENTRALIZED_FP32:
    print("\n--- Running Centralized FP32 ---")
    model = EndpointMLP().to(DEVICE)
    train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=BATCH, shuffle=True)
    
    local_train(model, train_loader, epochs=EPOCHS, use_prox=False)
    _, acc, preds, labels = evaluate_model(model, test_loader)
    metrics = evaluate_metrics(labels, preds)
    mem, lat = estimate_model_metrics(model, is_quantized=False)
    metrics.update({'Memory (KB)': mem, 'Latency (ms)': lat, 'Accuracy': acc})
    results_db['Centralized FP32'] = metrics
    
    # Store dummy convergence
    convergence_history['Centralized FP32'] = [metrics['f1_macro']] * ROUNDS
    print(metrics)

# Federated Setup
client_indices = dirichlet_partition(y_train, N_CLIENTS, alpha=NON_IID_ALPHA)
client_loaders = []
for idx in client_indices:
    if len(idx) == 0: continue
    ds = TensorDataset(torch.tensor(X_train[idx]), torch.tensor(y_train[idx]))
    client_loaders.append(DataLoader(ds, batch_size=BATCH, shuffle=True))

def run_federated_loop(name, use_prox, use_quant, use_trust_agg, use_dp, use_bces):
    print(f"\n--- Running {name} ---")
    global_model = EndpointMLP().to(DEVICE)
    round_f1s = []
    
    global client_reliability
    client_reliability = {i: 0.5 for i in range(N_CLIENTS)}
    
    for r in range(ROUNDS):
        client_weights = []
        client_metrics = []
        client_sizes = []
        
        for i in range(len(client_loaders)):
            client_model = copy.deepcopy(global_model)
            w = local_train(client_model, client_loaders[i], epochs=LOCAL_EPOCHS, use_prox=use_prox, global_model=global_model, use_dp=use_dp)
            client_weights.append(w)
            client_sizes.append(len(client_loaders[i].dataset))
            
            if use_trust_agg:
                _, acc, _, _ = evaluate_model(client_model, client_loaders[i])
                client_metrics.append({'val_acc': acc})
                
        if use_trust_agg:
            global_model = trust_aware_aggregation(global_model, client_weights, client_metrics, r)
        else:
            global_model = fedavg_aggregation(global_model, client_weights, client_sizes)
            
        # Log convergence
        _, _, preds, labels = evaluate_model(global_model, test_loader)
        round_f1s.append(f1_score(labels, preds, average='macro'))
        
    convergence_history[name] = round_f1s
    
    if use_dp and OPACUS_AVAILABLE and global_accountant is not None:
        epsilon = global_accountant.get_epsilon(delta=1e-5)
        print(f"Privacy Budget Exhausted: epsilon={epsilon:.2f}, delta=1e-5")
            
    if use_quant:
        eval_model = quantize_model(global_model)
    else:
        eval_model = global_model
        
    _, acc, preds, labels = evaluate_model(eval_model, test_loader)
    metrics = evaluate_metrics(labels, preds)
    mem, lat = estimate_model_metrics(eval_model, is_quantized=use_quant)
    
    if use_bces:
        scheduler = BCESScheduler(budget_ms=200.0)
        anomaly_idx = np.where(y_test > 0)[0][:100]
        events = [(torch.tensor(X_test[i]), y_test[i]) for i in anomaly_idx]
        selected = scheduler.schedule(events, eval_model)
        xai_overhead = simulate_xai_overhead(events, selected)
        metrics['XAI Cost (ms)'] = xai_overhead
        metrics['Events Explained'] = len(selected)
    
    metrics.update({'Memory (KB)': mem, 'Latency (ms)': lat, 'Accuracy': acc})
    results_db[name] = metrics
    print(metrics)

if RUN_FEDAVG_INT8:
    run_federated_loop('FedAvg INT8', use_prox=False, use_quant=True, use_trust_agg=False, use_dp=False, use_bces=False)

if RUN_FEDPROX_FP32:
    run_federated_loop('FedProx FP32', use_prox=True, use_quant=False, use_trust_agg=False, use_dp=False, use_bces=False)

if RUN_FEDX_TINYIDS:
    run_federated_loop('FedX-TinyIDS', use_prox=True, use_quant=True, use_trust_agg=True, use_dp=OPACUS_AVAILABLE, use_bces=True)

if RUN_ABLATION_TRUST:
    run_federated_loop('FedX (No Trust Agg)', use_prox=True, use_quant=True, use_trust_agg=False, use_dp=OPACUS_AVAILABLE, use_bces=True)

if RUN_ABLATION_DP:
    print("\n--- DP Sweep Ablation ---")
    old_noise = DP_NOISE_MULTIPLIER
    for n in [0.5, 1.0, 1.5]:
        DP_NOISE_MULTIPLIER = n
        run_federated_loop(f'FedX DP Sigma={n}', use_prox=True, use_quant=True, use_trust_agg=True, use_dp=OPACUS_AVAILABLE, use_bces=False)
    DP_NOISE_MULTIPLIER = old_noise

print("\n==========================================")
print("             Final Results")
print("==========================================")
res_df = pd.DataFrame(results_db).T
print(res_df.to_string())


## Visualizations

In [ ]:
# Convergence Plot
plt.figure(figsize=(10, 6))
for name, history in convergence_history.items():
    plt.plot(range(1, len(history)+1), history, label=name, linewidth=2)
plt.title("Federated Convergence (Macro-F1) over Global Rounds")
plt.xlabel("Global Round")
plt.ylabel("Macro-F1 Score")
plt.legend()
plt.tight_layout()
plt.show()

# Memory and Latency Profiling
if len(results_db) > 0:
    plot_df = pd.DataFrame(results_db).T.reset_index()
    plot_df = plot_df.rename(columns={'index': 'Model'})
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.barplot(data=plot_df, x='Model', y='Memory (KB)', ax=axes[0], palette='viridis')
    axes[0].set_title('Endpoint Memory Footprint (KB)')
    axes[0].tick_params(axis='x', rotation=45)
    
    sns.barplot(data=plot_df, x='Model', y='Latency (ms)', ax=axes[1], palette='magma')
    axes[1].set_title('Endpoint Inference Latency (ms)')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
